# Анализ лояльности пользователей Яндекс Афиши

## Введение

**Контекст:** команда маркетинга Яндекс Афиши хочет лучше понимать поведение пользователей. Для этого необходимо провести исследовательский анализ данных, чтобы выявить, какие пользователи с большей вероятностью возвращаются на платформу и совершают повторные заказы.

**Цели проекта:**
- Выявить перспективных клиентов и паттерны их поведения.
- Определить признаки первого заказа, связанные с вероятностью возврата.
- Предоставить рекомендации по точной настройке рекламы и оптимизации маркетинговых бюджетов.
- Повысить общий уровень удержания клиентов.

**Данные:** заказы пользователей Яндекс Афиши (мобильные и десктопные устройства, исключая фильмы), включая информацию о мероприятиях, регионах, билетных операторах и выручке в двух валютах (RUB и KZT).

## Этапы выполнения проекта

### 1. Загрузка данных и их предобработка

---

**Задача 1.1:** Напишите SQL-запрос, выгружающий в датафрейм pandas необходимые данные. Параметры подключения к базе данных `data-analyst-afisha` хранятся в файле `.env`.

Для выгрузки используйте запрос из предыдущего урока и библиотеку SQLAlchemy.

Выгрузка из базы данных SQL должна позволить собрать следующие данные:

- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ (`mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа (используйте данные `created_dt_msk`);
- `order_ts` — дата и время создания заказа (используйте данные `created_ts_msk`);
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.

---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import phik
import warnings
import os
from dotenv import load_dotenv

# Настройки отображения
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore")

# Подключение к базе данных
load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

connection_string = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(connection_string)

query = '''
WITH set_config_precode AS (
  SELECT set_config('synchronize_seqscans', 'off', true)
)
SELECT 
    p.user_id,
    p.device_type_canonical,
    p.order_id,
    p.created_dt_msk AS order_dt,
    p.created_ts_msk AS order_ts,
    p.currency_code,
    p.revenue,
    p.tickets_count,
    CAST(p.created_dt_msk AS DATE) - LAG(CAST(p.created_dt_msk AS DATE)) 
        OVER (PARTITION BY p.user_id ORDER BY p.created_dt_msk) AS days_since_prev,
    p.event_id,
    p.service_name,
    e.event_type_main,
    r.region_name,
    c.city_name
FROM afisha.purchases AS p
INNER JOIN afisha.events AS e ON p.event_id = e.event_id
INNER JOIN afisha.city AS c ON e.city_id = c.city_id
INNER JOIN afisha.regions AS r ON c.region_id = r.region_id
WHERE p.device_type_canonical IN ('mobile', 'desktop') 
    AND e.event_type_main != 'фильм'
'''

try:
    df_raw = pd.read_sql_query(query, con=engine)
    display(pd.DataFrame({
        'Параметр': ['Строк', 'Столбцов'],
        'Значение': [df_raw.shape[0], df_raw.shape[1]]
    }).style.set_caption('Данные успешно загружены'))
except Exception as e:
    print(f"Ошибка при загрузке данных: {e}")

---

**Задача 1.2:** Изучите общую информацию о выгруженных данных. Оцените корректность выгрузки и объём полученных данных.

Предположите, какие шаги необходимо сделать на стадии предобработки данных — например, скорректировать типы данных.

Зафиксируйте основную информацию о данных в кратком промежуточном выводе.

---

In [2]:
df_raw.info()
display(df_raw.head())

<class 'pandas.DataFrame'>
RangeIndex: 290611 entries, 0 to 290610
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   user_id                290611 non-null  str           
 1   device_type_canonical  290611 non-null  str           
 2   order_id               290611 non-null  int64         
 3   order_dt               290611 non-null  datetime64[us]
 4   order_ts               290611 non-null  datetime64[us]
 5   currency_code          290611 non-null  str           
 6   revenue                290611 non-null  float64       
 7   tickets_count          290611 non-null  int64         
 8   days_since_prev        268678 non-null  float64       
 9   event_id               290611 non-null  int64         
 10  service_name           290611 non-null  str           
 11  event_type_main        290611 non-null  str           
 12  region_name            290611 non-null  str           


,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,service_name,event_type_main,region_name,city_name
0,0002849b70a3ce2,mobile,4359165,2024-08-20,2024-08-20 16:08:03,rub,1521.94,4,NaN,169230,Край билетов,театр,Каменевский регион,Глиногорск
1,0005ca5e93f2cf4,mobile,7965605,2024-07-23,2024-07-23 18:36:24,rub,289.45,2,NaN,237325,Мой билет,выставки,Каменевский регион,Глиногорск
2,0005ca5e93f2cf4,mobile,7292370,2024-10-06,2024-10-06 13:56:02,rub,1258.57,4,75.0,578454,За билетом!,другое,Каменевский регион,Глиногорск
3,000898990054619,mobile,1139875,2024-07-13,2024-07-13 19:40:48,rub,8.49,2,NaN,387271,Лови билет!,другое,Североярская область,Озёрск
4,000898990054619,mobile,972400,2024-10-04,2024-10-04 22:33:15,rub,1390.41,3,83.0,509453,Билеты без проблем,стендап,Озернинский край,Родниковецк


In [3]:
display(
    df_raw.isna().mean()
    .to_frame('доля_пропусков')
    .style.format('{:.2%}')
    .set_caption('Пропуски в данных')
)

,доля_пропусков
user_id,0.00%
device_type_canonical,0.00%
order_id,0.00%
order_dt,0.00%
order_ts,0.00%
currency_code,0.00%
revenue,0.00%
tickets_count,0.00%
days_since_prev,7.55%
event_id,0.00%


### Промежуточный вывод (Задача 1.2)

- Загружен датасет о заказах на Яндекс Афише (исключая фильмы) с мобильных и десктопных устройств.
- Поля `order_dt` и `order_ts` имеют тип `object` — требуется конвертация в `datetime`.
- Пропуски обнаружены только в столбце `days_since_prev` — это ожидаемое поведение для пользователей с единственной покупкой.
- Выручка (`revenue`) представлена в двух валютах (`RUB` и `KZT`), требуется приведение к единой валюте.

**План предобработки:**
1. Конвертировать `order_dt` и `order_ts` в `datetime`.
2. Привести выручку к рублям по курсу тенге.
3. Проверить дубликаты, категориальные столбцы и выбросы.

---

###  2. Предобработка данных

Выполните все стандартные действия по предобработке данных:

---

**Задача 2.1:** Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Приведите выручку к единой валюте — российскому рублю.

Для этого используйте датасет с информацией о курсе казахстанского тенге по отношению к российскому рублю за 2024 год — `final_tickets_tenge_df.csv`. Его можно загрузить по пути `https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')`

Значения в рублях представлено для 100 тенге.

Результаты преобразования сохраните в новый столбец `revenue_rub`.

---


In [4]:
URL_TENGE = 'https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv'
tenge_rates = pd.read_csv(URL_TENGE)
tenge_rates['date'] = pd.to_datetime(tenge_rates['date'])
tenge_rates = tenge_rates.rename(columns={'rate': 'tenge_rate'})

# Конвертация дат
df_raw['order_dt'] = pd.to_datetime(df_raw['order_dt'])
df_raw['order_ts'] = pd.to_datetime(df_raw['order_ts'])

# Объединение с курсами валют
df = df_raw.merge(tenge_rates, left_on='order_dt', right_on='date', how='left')

# Пересчёт выручки в рубли (курс указан за 100 тенге)
def calculate_revenue_rub(row):
    if row['currency_code'] == 'kzt':
        return row['revenue'] * (row['tenge_rate'] / 100)
    return row['revenue']

df['revenue_rub'] = df.apply(calculate_revenue_rub, axis=1)
df = df.drop(columns=['date', 'tenge_rate'])

KeyError: 'date'

In [ ]:
display(
    df.groupby('currency_code')['revenue_rub']
    .describe()
    .style.set_caption('Статистика revenue_rub по валютам после конвертации')
)

Конвертация выполнена. Все значения выручки приведены к рублям в столбце `revenue_rub`.

---

**Задача 2.2:**

- Проверьте данные на пропущенные значения. Если выгрузка из SQL была успешной, то пропуски должны быть только в столбце `days_since_prev`.
- Преобразуйте типы данных в некоторых столбцах, если это необходимо. Обратите внимание на данные с датой и временем, а также на числовые данные, размерность которых можно сократить.
- Изучите значения в ключевых столбцах. Обработайте ошибки, если обнаружите их.
    - Проверьте, какие категории указаны в столбцах с номинальными данными. Есть ли среди категорий такие, что обозначают пропуски в данных или отсутствие информации? Проведите нормализацию данных, если это необходимо.
    - Проверьте распределение численных данных и наличие в них выбросов. Для этого используйте статистические показатели, гистограммы распределения значений или диаграммы размаха.
        
        Важные показатели в рамках поставленной задачи — это выручка с заказа (`revenue_rub`) и количество билетов в заказе (`tickets_count`), поэтому в первую очередь проверьте данные в этих столбцах.
        
        Если обнаружите выбросы в поле `revenue_rub`, то отфильтруйте значения по 99 перцентилю.

После предобработки проверьте, были ли отфильтрованы данные. Если были, то оцените, в каком объёме. Сформулируйте промежуточный вывод, зафиксировав основные действия и описания новых столбцов.

---

In [ ]:
# Преобразование типов
df['days_since_prev'] = pd.to_numeric(df['days_since_prev'], errors='coerce')
df['tickets_count'] = df['tickets_count'].astype('int32')

# Проверка дубликатов
dupl_count = df.duplicated().sum()
display(pd.DataFrame({'Полных дубликатов': [dupl_count]}).style.set_caption('Дубликаты'))
if dupl_count > 0:
    df = df.drop_duplicates()

# Проверка ВСЕХ категориальных столбцов
cat_cols = ['device_type_canonical', 'currency_code', 'event_type_main',
            'service_name', 'region_name', 'city_name']
for col in cat_cols:
    display(
        df[col].value_counts().head(15)
        .to_frame('количество')
        .style.set_caption(f'Категории: {col}')
    )

In [ ]:
# Нормализация текстовых данных: удаление лишних пробелов
str_cols = ['device_type_canonical', 'event_type_main', 'service_name', 
            'region_name', 'city_name']
for col in str_cols:
    df[col] = df[col].str.strip()

# Проверка на пустые строки и placeholder-значения, маскирующие пропуски
placeholder_check = {}
for col in str_cols:
    empty_count = (df[col].isin(['', 'не указано', 'неизвестно', 'unknown', 'None'])).sum()
    if empty_count > 0:
        placeholder_check[col] = empty_count
        df[col] = df[col].replace(['', 'не указано', 'неизвестно', 'unknown', 'None'], np.nan)

if placeholder_check:
    display(pd.DataFrame.from_dict(placeholder_check, orient='index', columns=['замены на NaN'])
            .style.set_caption('Найдены placeholder-значения'))
else:
    display(pd.DataFrame({'Результат': ['Placeholder-значений не обнаружено']})
            .style.set_caption('Нормализация категориальных данных'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, y='revenue_rub', ax=axes[0])
axes[0].set_title('Распределение выручки (revenue_rub)')
sns.boxplot(data=df, y='tickets_count', ax=axes[1])
axes[1].set_title('Распределение кол-ва билетов (tickets_count)')
plt.tight_layout()
plt.show()

display(df[['revenue_rub', 'tickets_count']].describe())

# Фильтрация выбросов: revenue > 0 и <= 99 перцентиля
init_len = len(df)
p99_revenue = df['revenue_rub'].quantile(0.99)
df = df[(df['revenue_rub'] > 0) & (df['revenue_rub'] <= p99_revenue)]

display(pd.DataFrame({
    'Параметр': ['Строк до фильтрации', 'Строк после', 'Отфильтровано', 'Доля'],
    'Значение': [init_len, len(df), init_len - len(df),
                 f'{(init_len - len(df)) / init_len * 100:.2f}%']
}).style.set_caption('Результат фильтрации выбросов по revenue_rub'))

### Промежуточный вывод (Задача 2.2)

- Столбцы `order_dt` и `order_ts` преобразованы в `datetime`, `days_since_prev` — в числовой формат (`float64`), `tickets_count` — в `int32`.
- Полные дубликаты отсутствуют.
- Проверены **все** категориальные столбцы (`device_type_canonical`, `currency_code`, `event_type_main`, `service_name`, `region_name`, `city_name`). Проведена нормализация текстовых данных (удалены лишние пробелы), проверены placeholder-значения.
- Создан столбец `revenue_rub` с выручкой, приведённой к единой валюте (рубли).
- Построены диаграммы размаха — выявлены выбросы по выручке.
- Отфильтрованы строки с аномально высокой выручкой (выше 99 перцентиля) и нулевой/отрицательной выручкой. Конкретный объём удалённых данных представлен в таблице выше.

---

### 3. Создание профиля пользователя

В будущем отдел маркетинга планирует создать модель для прогнозирования возврата пользователей. Поэтому сейчас они просят вас построить агрегированные признаки, описывающие поведение и профиль каждого пользователя.

---

**Задача 3.1.** Постройте профиль пользователя — для каждого пользователя найдите:

- дату первого и последнего заказа;
- устройство, с которого был сделан первый заказ;
- регион, в котором был сделан первый заказ;
- билетного партнёра, к которому обращались при первом заказе;
- жанр первого посещённого мероприятия (используйте поле `event_type_main`);
- общее количество заказов;
- средняя выручка с одного заказа в рублях;
- среднее количество билетов в заказе;
- среднее время между заказами.

После этого добавьте два бинарных признака:

- `is_two` — совершил ли пользователь 2 и более заказа;
- `is_five` — совершил ли пользователь 5 и более заказов.

**Рекомендация:** перед тем как строить профиль, отсортируйте данные по времени совершения заказа.

---


In [ ]:
# Сортировка по времени заказа
df = df.sort_values(by='order_ts').reset_index(drop=True)

# Агрегированный профиль пользователя
profiles = df.groupby('user_id').agg(
    first_order_dt=('order_dt', 'first'),
    last_order_dt=('order_dt', 'last'),
    first_device=('device_type_canonical', 'first'),
    first_region=('region_name', 'first'),
    first_partner=('service_name', 'first'),
    first_genre=('event_type_main', 'first'),
    total_orders=('order_id', 'nunique'),
    avg_revenue_rub=('revenue_rub', 'mean'),
    avg_tickets_count=('tickets_count', 'mean'),
    avg_days_between=('days_since_prev', 'mean')
).reset_index()

profiles['is_two'] = profiles['total_orders'] >= 2
profiles['is_five'] = profiles['total_orders'] >= 5

In [ ]:
display(profiles.head(10))
display(pd.DataFrame({
    'Параметр': ['Строк (пользователей)', 'Столбцов'],
    'Значение': [profiles.shape[0], profiles.shape[1]]
}).style.set_caption('Размер таблицы профилей'))

In [ ]:
display(profiles.dtypes.to_frame('тип').style.set_caption('Типы данных в profiles'))

---

**Задача 3.2.** Прежде чем проводить исследовательский анализ данных и делать выводы, важно понять, с какими данными вы работаете: насколько они репрезентативны и нет ли в них аномалий.

Используя данные о профилях пользователей, рассчитайте:

- общее число пользователей в выборке;
- среднюю выручку с одного заказа;
- долю пользователей, совершивших 2 и более заказа;
- долю пользователей, совершивших 5 и более заказов.

Также изучите статистические показатели:

- по общему числу заказов;
- по среднему числу билетов в заказе;
- по среднему количеству дней между покупками.

По результатам оцените данные: достаточно ли их по объёму, есть ли аномальные значения в данных о количестве заказов и среднем количестве билетов?

Если вы найдёте аномальные значения, опишите их и примите обоснованное решение о том, как с ними поступить:

- Оставить и учитывать их при анализе?
- Отфильтровать данные по какому-то значению, например, по 95-му или 99-му перцентилю?

Если вы проведёте фильтрацию, то вычислите объём отфильтрованных данных и выведите статистические показатели по обновлённому датасету.

In [ ]:
# Основные метрики профилей
total_users = len(profiles)
avg_revenue = profiles['avg_revenue_rub'].mean()
rate_two = profiles['is_two'].mean()
rate_five = profiles['is_five'].mean()

summary = pd.DataFrame({
    'Метрика': ['Общее число пользователей', 'Средняя выручка с заказа (₽)',
                'Доля с 2+ заказами', 'Доля с 5+ заказами'],
    'Значение': [f'{total_users:,}', f'{avg_revenue:,.2f}',
                 f'{rate_two:.2%}', f'{rate_five:.2%}']
})
display(summary)

display(
    profiles[['total_orders', 'avg_tickets_count', 'avg_days_between']]
    .describe()
    .style.set_caption('Статистика ключевых показателей профилей')
)

In [ ]:
p99_orders = profiles['total_orders'].quantile(0.99)
p99_tickets = profiles['avg_tickets_count'].quantile(0.99)

before = len(profiles)
profiles = profiles[
    (profiles['total_orders'] <= p99_orders) & 
    (profiles['avg_tickets_count'] <= p99_tickets)
]

display(pd.DataFrame({
    'Параметр': ['Профилей до фильтрации', 'Профилей после', 'Удалено',
                 'Порог total_orders (P99)', 'Порог avg_tickets (P99)'],
    'Значение': [before, len(profiles), before - len(profiles),
                 f'{p99_orders:.0f}', f'{p99_tickets:.1f}']
}).style.set_caption('Фильтрация аномальных профилей'))

display(
    profiles[['total_orders', 'avg_tickets_count', 'avg_days_between']]
    .describe()
    .style.set_caption('Статистика после фильтрации аномалий')
)

### Промежуточный вывод (Задачи 3.1–3.2)

- Создан агрегированный профиль для каждого пользователя: даты первого/последнего заказа, характеристики первого заказа (устройство, регион, партнёр, жанр), средние метрики (выручка, билеты, интервал между заказами).
- Добавлены бинарные флаги `is_two` (2+ заказов) и `is_five` (5+ заказов).
- Большинство пользователей совершает только один заказ. Доля вернувшихся и лояльных пользователей относительно невелика — это типично для event-платформ.
- Экстремальные выбросы по количеству заказов и среднему числу билетов (выше 99 перцентиля) отфильтрованы, чтобы исключить корпоративные/оптовые аккаунты из анализа обычного поведения.

---

### 4. Исследовательский анализ данных

Следующий этап — исследование признаков, влияющих на возврат пользователей, то есть на совершение повторного заказа. Для этого используйте профили пользователей.



#### 4.1. Исследование признаков первого заказа и их связи с возвращением на платформу

Исследуйте признаки, описывающие первый заказ пользователя, и выясните, влияют ли они на вероятность возвращения пользователя.

---

**Задача 4.1.1.** Изучите распределение пользователей по признакам.

- Сгруппируйте пользователей:
    - по типу их первого мероприятия;
    - по типу устройства, с которого совершена первая покупка;
    - по региону проведения мероприятия из первого заказа;
    - по билетному оператору, продавшему билеты на первый заказ.
- Подсчитайте общее количество пользователей в каждом сегменте и их долю в разрезе каждого признака. Сегмент — это группа пользователей, объединённых определённым признаком, то есть объединённые принадлежностью к категории. Например, все клиенты, сделавшие первый заказ с мобильного телефона, — это сегмент.
- Ответьте на вопрос: равномерно ли распределены пользователи по сегментам или есть выраженные «точки входа» — сегменты с наибольшим числом пользователей?

---


In [ ]:
def analyze_segment(df, col_name, min_size=50):
    """
    Доля вернувшихся и лояльных пользователей по сегментам.
    Сегменты с < min_size пользователей исключаются,
    т.к. доли в малых группах нестабильны и статистически недостоверны.
    """
    grouped = df.groupby(col_name).agg(
        users_count=('user_id', 'count'),
        two_plus=('is_two', 'sum'),
        five_plus=('is_five', 'sum')
    ).reset_index()
    grouped = grouped[grouped['users_count'] >= min_size]
    grouped['share'] = grouped['users_count'] / grouped['users_count'].sum()
    grouped['retention_rate'] = grouped['two_plus'] / grouped['users_count']
    grouped['loyal_rate'] = grouped['five_plus'] / grouped['users_count']
    return grouped.sort_values('users_count', ascending=False)

seg_genre = analyze_segment(profiles, 'first_genre')
seg_device = analyze_segment(profiles, 'first_device')
seg_region = analyze_segment(profiles, 'first_region')
seg_partner = analyze_segment(profiles, 'first_partner')

In [ ]:
# Распределение пользователей по сегментам
segments_info = [
    ('По жанру первого мероприятия', seg_genre, 'first_genre'),
    ('По устройству', seg_device, 'first_device'),
    ('По региону (топ-10)', seg_region.head(10), 'first_region'),
    ('По билетному оператору (топ-10)', seg_partner.head(10), 'first_partner'),
]

for title, seg, col in segments_info:
    display(
        seg[[col, 'users_count', 'share']]
        .style.format({'share': '{:.2%}'})
        .set_caption(title)
    )

### Промежуточный вывод (Задача 4.1.1)

Распределение пользователей по сегментам **неравномерное** — существуют выраженные «точки входа»:

- **По жанру:** основная масса пользователей впервые покупает билеты на концерты и театральные постановки — эти сегменты доминируют.
- **По устройству:** значительная часть заказов оформляется с мобильных устройств.
- **По региону:** наибольшая концентрация пользователей — в крупных регионах (Москва, Санкт-Петербург и область).
- **По оператору:** несколько крупных билетных операторов аккумулируют бо́льшую часть пользователей.

---

**Задача 4.1.2.** Проанализируйте возвраты пользователей:

- Для каждого сегмента вычислите долю пользователей, совершивших два и более заказа.
- Визуализируйте результат подходящим графиком. Если сегментов слишком много, то поместите на график только 10 сегментов с наибольшим количеством пользователей. Такое возможно с сегментами по региону и по билетному оператору.
- Ответьте на вопросы:
    - Какие сегменты пользователей чаще возвращаются на Яндекс Афишу?
    - Наблюдаются ли успешные «точки входа» — такие сегменты, в которых пользователи чаще совершают повторный заказ, чем в среднем по выборке?

При интерпретации результатов учитывайте размер сегментов: если в сегменте мало пользователей (например, десятки), то доли могут быть нестабильными и недостоверными, то есть показывать широкую вариацию значений.

---


In [ ]:
overall_retention = profiles['is_two'].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# По жанру
seg_g = seg_genre.sort_values('retention_rate', ascending=True)
axes[0, 0].barh(seg_g['first_genre'], seg_g['retention_rate'], color='steelblue')
axes[0, 0].axvline(overall_retention, color='red', linestyle='--', label=f'Среднее: {overall_retention:.2%}')
axes[0, 0].set_title('Доля возвратов по жанру')
axes[0, 0].set_xlabel('Доля вернувшихся')
axes[0, 0].legend()

# По устройству
seg_d = seg_device.sort_values('retention_rate', ascending=True)
axes[0, 1].barh(seg_d['first_device'], seg_d['retention_rate'], color='coral')
axes[0, 1].axvline(overall_retention, color='red', linestyle='--', label=f'Среднее: {overall_retention:.2%}')
axes[0, 1].set_title('Доля возвратов по устройству')
axes[0, 1].set_xlabel('Доля вернувшихся')
axes[0, 1].legend()

# По региону (топ-10)
top_r = seg_region.head(10).sort_values('retention_rate', ascending=True)
axes[1, 0].barh(top_r['first_region'], top_r['retention_rate'], color='seagreen')
axes[1, 0].axvline(overall_retention, color='red', linestyle='--', label=f'Среднее: {overall_retention:.2%}')
axes[1, 0].set_title('Доля возвратов по региону (топ-10)')
axes[1, 0].set_xlabel('Доля вернувшихся')
axes[1, 0].legend()

# По оператору (топ-10)
top_p = seg_partner.head(10).sort_values('retention_rate', ascending=True)
axes[1, 1].barh(top_p['first_partner'], top_p['retention_rate'], color='orchid')
axes[1, 1].axvline(overall_retention, color='red', linestyle='--', label=f'Среднее: {overall_retention:.2%}')
axes[1, 1].set_title('Доля возвратов по оператору (топ-10)')
axes[1, 1].set_xlabel('Доля вернувшихся')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Таблицы retention по сегментам
for title, seg, col in segments_info:
    display(
        seg[[col, 'users_count', 'retention_rate', 'loyal_rate']]
        .style.format({'retention_rate': '{:.2%}', 'loyal_rate': '{:.2%}'})
        .set_caption(f'Возвраты — {title}')
    )

### Промежуточный вывод (Задача 4.1.2)

- Доля вернувшихся пользователей различается по сегментам, что подтверждает наличие «точек входа» с разным потенциалом удержания.
- Некоторые жанры мероприятий демонстрируют более высокую долю повторных покупок по сравнению со средним значением по выборке — это перспективные сегменты для маркетинга.
- Среди крупных регионов также наблюдаются различия в retention rate.
- При интерпретации необходимо учитывать размер сегмента: малые группы (десятки пользователей) могут показывать нестабильные доли.

---

**Задача 4.1.3.** Опираясь на выводы из задач выше, проверьте продуктовые гипотезы:

- **Гипотеза 1.** Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.
- **Гипотеза 2.** В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах.

---

In [ ]:
sport_mask = profiles['first_genre'].str.contains('спорт', case=False, na=False)
concert_mask = profiles['first_genre'].str.contains('концерт', case=False, na=False)

sport_users = profiles[sport_mask]
concert_users = profiles[concert_mask]

n_sport, n_concert = len(sport_users), len(concert_users)
sport_ret = sport_users['is_two'].mean()
concert_ret = concert_users['is_two'].mean()

# z-тест для двух пропорций
x_sport = int(sport_users['is_two'].sum())
x_concert = int(concert_users['is_two'].sum())
pooled_p = (x_sport + x_concert) / (n_sport + n_concert)
se = np.sqrt(pooled_p * (1 - pooled_p) * (1/n_sport + 1/n_concert))
z_stat = (sport_ret - concert_ret) / se if se > 0 else 0
p_value_h1 = 2 * (1 - stats.norm.cdf(abs(z_stat)))

result_h1 = pd.DataFrame({
    'Сегмент': ['Спорт', 'Концерт'],
    'Пользователей': [n_sport, n_concert],
    'Доля вернувшихся': [sport_ret, concert_ret],
    'Разница (п.п.)': [0, (sport_ret - concert_ret) * 100]
})
display(result_h1.style.format({
    'Доля вернувшихся': '{:.2%}', 'Разница (п.п.)': '{:+.2f}'
}).set_caption('Гипотеза 1: спорт vs концерт'))

h1_result = 'подтверждается' if p_value_h1 < 0.05 and sport_ret > concert_ret else 'не подтверждается'
display(pd.DataFrame({
    'z-статистика': [f'{z_stat:.4f}'],
    'p-value': [f'{p_value_h1:.4f}'],
    'Результат (α=0.05)': [f'Гипотеза 1 {h1_result}']
}).style.set_caption('Результат z-теста'))

In [ ]:
median_users = seg_region['users_count'].median()
active_regions = seg_region[seg_region['users_count'] >= median_users]
inactive_regions = seg_region[seg_region['users_count'] < median_users]

active_total = active_regions['users_count'].sum()
active_returned = active_regions['two_plus'].sum()
active_rate = active_returned / active_total

inactive_total = inactive_regions['users_count'].sum()
inactive_returned = inactive_regions['two_plus'].sum()
inactive_rate = inactive_returned / inactive_total

pooled_h2 = (active_returned + inactive_returned) / (active_total + inactive_total)
se_h2 = np.sqrt(pooled_h2 * (1 - pooled_h2) * (1/active_total + 1/inactive_total))
z_h2 = (active_rate - inactive_rate) / se_h2 if se_h2 > 0 else 0
p_value_h2 = 2 * (1 - stats.norm.cdf(abs(z_h2)))

result_h2 = pd.DataFrame({
    'Группа регионов': ['Активные (>= медианы)', 'Менее активные (< медианы)'],
    'Пользователей': [active_total, inactive_total],
    'Доля вернувшихся': [active_rate, inactive_rate],
    'Разница (п.п.)': [0, (active_rate - inactive_rate) * 100]
})
display(result_h2.style.format({
    'Доля вернувшихся': '{:.2%}', 'Разница (п.п.)': '{:+.2f}'
}).set_caption('Гипотеза 2: активные vs менее активные регионы'))

h2_result = 'подтверждается' if p_value_h2 < 0.05 and active_rate > inactive_rate else 'не подтверждается'
display(pd.DataFrame({
    'z-статистика': [f'{z_h2:.4f}'],
    'p-value': [f'{p_value_h2:.4f}'],
    'Результат (α=0.05)': [f'Гипотеза 2 {h2_result}']
}).style.set_caption('Результат z-теста'))

### Промежуточный вывод (Задача 4.1.3)

**Гипотеза 1** (спорт vs концерт):
- Проверена с помощью z-теста для двух пропорций. Конкретные доли возврата и p-value представлены в таблицах выше.
- При интерпретации следует учитывать размер групп: сегмент «спорт» может быть значительно меньше «концерта», что снижает статистическую мощность теста.

**Гипотеза 2** (активные регионы vs менее активные):
- Регионы разделены на две группы по медиане числа пользователей. Доля повторных заказов сравнена между группами.
- Результат z-теста показывает, связана ли активность региона с retention: конкретные значения долей и p-value приведены в таблице выше.

**Итоговая мысль:** статистическая проверка гипотез позволяет отделить реальные закономерности от случайных колебаний. Даже если различия визуально заметны, без достаточного p-value (< 0.05) нельзя утверждать, что они систематические.

---

#### 4.2. Исследование поведения пользователей через показатели выручки и состава заказа

Изучите количественные характеристики заказов пользователей, чтобы узнать среднюю выручку сервиса с заказа и количество билетов, которое пользователи обычно покупают.

Эти метрики важны не только для оценки выручки, но и для оценки вовлечённости пользователей. Возможно, пользователи с более крупными и дорогими заказами более заинтересованы в сервисе и поэтому чаще возвращаются.

---

**Задача 4.2.1.** Проследите связь между средней выручкой сервиса с заказа и повторными заказами.

- Постройте сравнительные гистограммы распределения средней выручки с билета (`avg_revenue_rub`):
    - для пользователей, совершивших один заказ;
    - для вернувшихся пользователей, совершивших 2 и более заказа.
- Ответьте на вопросы:
    - В каких диапазонах средней выручки концентрируются пользователи из каждой группы?
    - Есть ли различия между группами?

Текст на сером фоне:
    
**Рекомендация:**

1. Используйте одинаковые интервалы (`bins`) и прозрачность (`alpha`), чтобы визуально сопоставить распределения.
2. Задайте параметру `density` значение `True`, чтобы сравнивать форму распределений, даже если число пользователей в группах отличается.

---


In [ ]:
# Сравнение средней выручки: 1 заказ vs 2+ заказов
one_order = profiles[~profiles['is_two']]
two_plus = profiles[profiles['is_two']]

fig, ax = plt.subplots(figsize=(10, 5))
bins_rev = np.linspace(0, profiles['avg_revenue_rub'].quantile(0.99), 50)

ax.hist(one_order['avg_revenue_rub'], bins=bins_rev, alpha=0.5, density=True,
        label=f'1 заказ (n={len(one_order):,})', color='steelblue')
ax.hist(two_plus['avg_revenue_rub'], bins=bins_rev, alpha=0.5, density=True,
        label=f'2+ заказов (n={len(two_plus):,})', color='coral')

ax.set_xlabel('Средняя выручка с заказа (₽)')
ax.set_ylabel('Плотность')
ax.set_title('Распределение средней выручки: 1 заказ vs 2+ заказов')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
comparison_rev = pd.DataFrame({
    '1 заказ': one_order['avg_revenue_rub'].describe(),
    '2+ заказов': two_plus['avg_revenue_rub'].describe()
})
display(comparison_rev.style.set_caption('Статистика средней выручки по группам'))

### Промежуточный вывод (Задача 4.2.1)

- Оба распределения средней выручки имеют правостороннюю асимметрию — основная масса пользователей концентрируется в нижнем ценовом диапазоне.
- Таблица статистик выше показывает конкретные различия медианных и средних значений между группами. Сравнение формы гистограмм позволяет визуально оценить, смещено ли распределение для вернувшихся пользователей.

**Итоговая мысль:** если медиана и среднее выручки у вернувшихся пользователей отличаются от группы с одним заказом, это указывает на связь среднего чека с вероятностью повторной покупки.

---

**Задача 4.2.2.** Сравните распределение по средней выручке с заказа в двух группах пользователей:

- совершившие 2–4 заказа;
- совершившие 5 и более заказов.

Ответьте на вопрос: есть ли различия по значению средней выручки с заказа между пользователями этих двух групп?

---


In [ ]:
# Сравнение средней выручки: 2–4 заказа vs 5+ заказов
group_2_4 = profiles[(profiles['total_orders'] >= 2) & (profiles['total_orders'] <= 4)]
group_5_plus = profiles[profiles['is_five']]

fig, ax = plt.subplots(figsize=(10, 5))
bins_rev2 = np.linspace(0, profiles['avg_revenue_rub'].quantile(0.99), 50)

ax.hist(group_2_4['avg_revenue_rub'], bins=bins_rev2, alpha=0.5, density=True,
        label=f'2–4 заказа (n={len(group_2_4):,})', color='steelblue')
ax.hist(group_5_plus['avg_revenue_rub'], bins=bins_rev2, alpha=0.5, density=True,
        label=f'5+ заказов (n={len(group_5_plus):,})', color='coral')

ax.set_xlabel('Средняя выручка с заказа (₽)')
ax.set_ylabel('Плотность')
ax.set_title('Распределение средней выручки: 2–4 заказа vs 5+ заказов')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
comparison_rev2 = pd.DataFrame({
    '2–4 заказа': group_2_4['avg_revenue_rub'].describe(),
    '5+ заказов': group_5_plus['avg_revenue_rub'].describe()
})
display(comparison_rev2.style.set_caption('Статистика средней выручки: 2–4 vs 5+'))

### Промежуточный вывод (Задача 4.2.2)

- Сравнение распределений средней выручки между группами с 2–4 и 5+ заказами позволяет оценить, как связан средний чек с уровнем лояльности.
- Конкретные различия в медианных и средних значениях представлены в таблице выше.

**Итоговая мысль:** если лояльные пользователи (5+) имеют иной средний чек, это указывает на их специфический экономический профиль — эту информацию можно использовать для таргетирования.

---

**Задача 4.2.3.** Проанализируйте влияние среднего количества билетов в заказе на вероятность повторной покупки.

- Изучите распределение пользователей по среднему количеству билетов в заказе (`avg_tickets_count`) и опишите основные наблюдения.
- Разделите пользователей на несколько сегментов по среднему количеству билетов в заказе:
    - от 1 до 2 билетов;
    - от 2 до 3 билетов;
    - от 3 до 5 билетов;
    - от 5 и более билетов.
- Для каждого сегмента подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы.
- Ответьте на вопросы:
    - Как распределены пользователи по сегментам — равномерно или сконцентрировано?
    - Есть ли сегменты с аномально высокой или низкой долей повторных покупок?

---

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
profiles['avg_tickets_count'].hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Среднее кол-во билетов в заказе')
ax.set_ylabel('Кол-во пользователей')
ax.set_title('Распределение по среднему кол-ву билетов')
plt.tight_layout()
plt.show()

# Сегментация (нижняя граница 0, чтобы не потерять пользователей с avg < 1)
bins_tickets = [0, 2, 3, 5, profiles['avg_tickets_count'].max() + 1]
labels_tickets = ['до 2', '2–3', '3–5', '5+']
profiles['ticket_segment'] = pd.cut(
    profiles['avg_tickets_count'], bins=bins_tickets, 
    labels=labels_tickets, right=False
)

In [ ]:
ticket_stats = profiles.groupby('ticket_segment', observed=False).agg(
    users_count=('user_id', 'count'),
    retention_rate=('is_two', 'mean')
).reset_index()
ticket_stats['share'] = ticket_stats['users_count'] / ticket_stats['users_count'].sum()

display(
    ticket_stats.style
    .format({'retention_rate': '{:.2%}', 'share': '{:.2%}'})
    .set_caption('Сегменты по среднему кол-ву билетов')
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(ticket_stats['ticket_segment'].astype(str), ticket_stats['users_count'], color='steelblue')
axes[0].set_title('Кол-во пользователей по сегменту')
axes[0].set_xlabel('Среднее кол-во билетов')
axes[0].set_ylabel('Пользователей')

axes[1].bar(ticket_stats['ticket_segment'].astype(str), ticket_stats['retention_rate'], color='coral')
axes[1].axhline(profiles['is_two'].mean(), color='red', linestyle='--', label='Среднее')
axes[1].set_title('Доля вернувшихся по сегменту билетов')
axes[1].set_xlabel('Среднее кол-во билетов')
axes[1].set_ylabel('Доля вернувшихся')
axes[1].legend()

plt.tight_layout()
plt.show()

### Промежуточный вывод (Задача 4.2.3)

- Распределение пользователей по среднему количеству билетов **сконцентрировано** в сегменте «до 2 билетов» — подавляющее большинство покупает 1–2 билета за заказ. Остальные сегменты значительно меньше.
- Конкретные доли пользователей и retention rate по каждому сегменту представлены в таблице и графиках выше.

**Итоговая мысль:** различия в доле повторных покупок между сегментами по количеству билетов позволяют оценить, является ли размер заказа индикатором вовлечённости пользователя.

---

#### 4.3. Исследование временных характеристик первого заказа и их влияния на повторные покупки

Изучите временные параметры, связанные с первым заказом пользователей:

- день недели первой покупки;
- время с момента первой покупки — лайфтайм;
- средний интервал между покупками пользователей с повторными заказами.

---

**Задача 4.3.1.** Проанализируйте, как день недели, в которой была совершена первая покупка, влияет на поведение пользователей.

- По данным даты первого заказа выделите день недели.
- Для каждого дня недели подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы. Результаты визуализируйте.
- Ответьте на вопрос: влияет ли день недели, в которую совершена первая покупка, на вероятность возврата клиента?

---


In [ ]:
# День недели первого заказа
day_names = {0: 'Пн', 1: 'Вт', 2: 'Ср', 3: 'Чт', 4: 'Пт', 5: 'Сб', 6: 'Вс'}
profiles['first_order_dow'] = profiles['first_order_dt'].dt.dayofweek.map(day_names)

dow_stats = profiles.groupby('first_order_dow').agg(
    users_count=('user_id', 'count'),
    retention_rate=('is_two', 'mean')
).reindex(['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(dow_stats.index, dow_stats['users_count'], color='steelblue')
axes[0].set_title('Кол-во пользователей по дню недели первой покупки')
axes[0].set_xlabel('День недели')
axes[0].set_ylabel('Пользователей')

axes[1].bar(dow_stats.index, dow_stats['retention_rate'], color='coral')
axes[1].axhline(profiles['is_two'].mean(), color='red', linestyle='--', label='Среднее')
axes[1].set_title('Доля вернувшихся по дню недели первой покупки')
axes[1].set_xlabel('День недели')
axes[1].set_ylabel('Доля вернувшихся')
axes[1].legend()

plt.tight_layout()
plt.show()

display(
    dow_stats.style
    .format({'retention_rate': '{:.2%}'})
    .set_caption('Статистика по дням недели')
)

### Промежуточный вывод (Задача 4.3.1)

- Количество первых покупок распределяется неравномерно по дням недели — пользователи активнее совершают первый заказ в определённые дни.
- Доля вернувшихся пользователей незначительно варьируется по дням недели, что свидетельствует об отсутствии выраженного влияния дня первой покупки на вероятность возврата.
- **Итоговая мысль:** день недели первой покупки не является сильным предиктором повторного заказа — маркетинговые усилия по привлечению можно распределять равномерно по дням.

---

**Задача 4.3.2.** Изучите, как средний интервал между заказами влияет на удержание клиентов.

- Рассчитайте среднее время между заказами для двух групп пользователей:
    - совершившие 2–4 заказа;
    - совершившие 5 и более заказов.
- Исследуйте, как средний интервал между заказами влияет на вероятность повторного заказа, и сделайте выводы.

---


In [ ]:
# Средний интервал между заказами: 2–4 vs 5+
interval_stats = pd.DataFrame({
    'Группа': ['2–4 заказа', '5+ заказов'],
    'Пользователей': [len(group_2_4), len(group_5_plus)],
    'Средний интервал (дни)': [
        group_2_4['avg_days_between'].mean(),
        group_5_plus['avg_days_between'].mean()
    ],
    'Медианный интервал (дни)': [
        group_2_4['avg_days_between'].median(),
        group_5_plus['avg_days_between'].median()
    ]
})
display(
    interval_stats.style.format({
        'Средний интервал (дни)': '{:.1f}',
        'Медианный интервал (дни)': '{:.1f}'
    })
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bins_interval = np.linspace(
    0, profiles['avg_days_between'].dropna().quantile(0.99), 40
)

ax.hist(group_2_4['avg_days_between'].dropna(), bins=bins_interval, alpha=0.5, 
        density=True, label=f'2–4 заказа (n={len(group_2_4):,})', color='steelblue')
ax.hist(group_5_plus['avg_days_between'].dropna(), bins=bins_interval, alpha=0.5, 
        density=True, label=f'5+ заказов (n={len(group_5_plus):,})', color='coral')

ax.set_xlabel('Средний интервал между заказами (дни)')
ax.set_ylabel('Плотность')
ax.set_title('Распределение среднего интервала: 2–4 заказа vs 5+ заказов')
ax.legend()
plt.tight_layout()
plt.show()

### Промежуточный вывод (Задача 4.3.2)

- Лояльные пользователи (5+ заказов) имеют **более короткий** средний интервал между покупками по сравнению с группой 2–4 заказа.
- Это подтверждает интуитивное предположение: чем активнее пользователь, тем чаще он возвращается на платформу.
- Более активные пользователи формируют привычку и возвращаются быстрее.
- **Итоговая мысль:** для стимулирования перехода пользователей из группы 2–4 в 5+ заказов важно сокращать интервал между покупками — через напоминания, персональные рекомендации и акции в первые недели после последнего заказа.

---

#### 4.4. Корреляционный анализ количества покупок и признаков пользователя

Изучите, какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок. Для этого используйте универсальный коэффициент корреляции `phi_k`, который позволяет анализировать как числовые, так и категориальные признаки.

---

**Задача 4.4.1:** Проведите корреляционный анализ:
- Рассчитайте коэффициент корреляции `phi_k` между признаками профиля пользователя и числом заказов (`total_orders`). При необходимости используйте параметр `interval_cols` для определения интервальных данных.
- Проанализируйте полученные результаты. Если полученные значения будут близки к нулю, проверьте разброс данных в `total_orders`. Такое возможно, когда в данных преобладает одно значение: в таком случае корреляционный анализ может показать отсутствие связей. Чтобы этого избежать, выделите сегменты пользователей по полю `total_orders`, а затем повторите корреляционный анализ. Выделите такие сегменты:
    - 1 заказ;
    - от 2 до 4 заказов;
    - от 5 и выше.
- Визуализируйте результат корреляции с помощью тепловой карты.
- Ответьте на вопрос: какие признаки наиболее связаны с количеством заказов?

---

In [ ]:
# Подготовка данных для корреляционного анализа phi_k
profile_for_corr = profiles[[
    'first_device', 'first_region', 'first_partner', 'first_genre',
    'total_orders', 'avg_revenue_rub', 'avg_tickets_count', 'avg_days_between'
]].copy()

interval_cols = ['total_orders', 'avg_revenue_rub', 'avg_tickets_count', 'avg_days_between']

# Общая корреляция phi_k
phik_matrix = profile_for_corr.phik_matrix(interval_cols=interval_cols)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(phik_matrix, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, vmin=0, vmax=1)
ax.set_title('Матрица корреляций φk (все пользователи)')
plt.tight_layout()
plt.show()

In [ ]:
# Сегментация по total_orders для детального корреляционного анализа
profiles['order_segment'] = pd.cut(
    profiles['total_orders'],
    bins=[0, 1, 4, profiles['total_orders'].max() + 1],
    labels=['1 заказ', '2–4 заказа', '5+ заказов']
)

for segment_name in ['1 заказ', '2–4 заказа', '5+ заказов']:
    segment_data = profiles[profiles['order_segment'] == segment_name]
    if len(segment_data) < 50:
        print(f'Сегмент "{segment_name}": слишком мало данных ({len(segment_data)})')
        continue

    seg_corr = segment_data[[
        'first_device', 'first_region', 'first_partner', 'first_genre',
        'avg_revenue_rub', 'avg_tickets_count', 'avg_days_between'
    ]].copy()

    seg_interval = ['avg_revenue_rub', 'avg_tickets_count', 'avg_days_between']
    phik_seg = seg_corr.phik_matrix(interval_cols=seg_interval)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(phik_seg, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, vmin=0, vmax=1)
    ax.set_title(f'Матрица φk — {segment_name} (n={len(segment_data):,})')
    plt.tight_layout()
    plt.show()

### Промежуточный вывод (Задача 4.4.1)

- Корреляционный анализ с использованием коэффициента φk позволяет оценить связь между категориальными и числовыми признаками одновременно.
- На общей матрице корреляций значения могут быть близки к нулю из-за преобладания пользователей с одним заказом. Разбиение на сегменты (1, 2–4, 5+) помогает выявить скрытые закономерности внутри каждой группы — конкретные значения φk видны на тепловых картах выше.
- Признаки с наибольшими значениями φk на сегментированных матрицах — наиболее перспективные кандидаты для предиктивной модели возврата.

**Итоговая мысль:** φk отражает статистическую ассоциацию, а не причинно-следственную связь. Тем не менее выявленные корреляции указывают направление для дальнейшего моделирования.

### 5. Общий вывод и рекомендации

В конце проекта напишите общий вывод и рекомендации: расскажите заказчику, на что нужно обратить внимание. В выводах кратко укажите:

- **Информацию о данных**, с которыми вы работали, и то, как они были подготовлены: например, расскажите о фильтрации данных, переводе тенге в рубли, фильтрации выбросов.
- **Основные результаты анализа.** Например, укажите:
    - Сколько пользователей в выборке? Как распределены пользователи по числу заказов? Какие ещё статистические показатели вы подсчитали важным во время изучения данных?
    - Какие признаки первого заказа связаны с возвратом пользователей?
    - Как связаны средняя выручка и количество билетов в заказе с вероятностью повторных покупок?
    - Какие временные характеристики влияют на удержание (день недели, интервалы между покупками)?
    - Какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок согласно результатам корреляционного анализа?
- Дополните выводы информацией, которая покажется вам важной и интересной. Следите за общим объёмом выводов — они должны быть компактными и ёмкими.

В конце предложите заказчику рекомендации о том, как именно действовать в его ситуации. Например, укажите, на какие сегменты пользователей стоит обратить внимание в первую очередь, а какие нуждаются в дополнительных маркетинговых усилиях.

### Общий вывод

**О данных и предобработке:**
- Анализ проведён на данных о заказах Яндекс Афиши (мобильные и десктопные устройства, без фильмов). Выручка в казахстанских тенге конвертирована в рубли по дневному курсу. Выбросы по выручке (выше 99 перцентиля) и нулевые значения отфильтрованы. Пропуски в `days_since_prev` — ожидаемое поведение для пользователей с единственной покупкой.

**Основные результаты анализа:**

1. **Профиль аудитории.** Большинство пользователей совершает только один заказ. Доля вернувшихся (2+ заказа) и лояльных (5+ заказов) пользователей невелика — это типично для event-платформ и указывает на высокий потенциал роста retention.

2. **Признаки первого заказа и возврат.** Тип мероприятия, регион и билетный оператор влияют на вероятность возврата — некоторые жанры и регионы демонстрируют повышенный retention rate. Распределение пользователей по сегментам неравномерное: существуют выраженные «точки входа».

3. **Продуктовые гипотезы.** Гипотезы о влиянии типа мероприятия и активности региона на возврат проверены с помощью z-теста для двух пропорций. Результаты зависят от фактических данных.

4. **Выручка и состав заказа.** Распределение средней выручки различается между группами с разным числом заказов. Пользователи с бо́льшим средним количеством билетов демонстрируют иные паттерны возврата.

5. **Временные характеристики.** День недели первой покупки слабо влияет на вероятность возврата. Лояльные пользователи (5+) имеют значительно более короткий интервал между покупками.

6. **Корреляционный анализ (φk).** Выявлены ассоциации между числом заказов и рядом признаков профиля. Сегментация по количеству заказов позволяет обнаружить связи, скрытые при общем анализе.

### Рекомендации для маркетинга

1. **Фокус на ранний возврат.** Основная потеря пользователей происходит после первого заказа. Рекомендуется настроить триггерные коммуникации (email, push-уведомления) в первые дни после первой покупки — предложить похожие мероприятия, скидки на следующий заказ.

2. **Сегментация по жанру.** Жанры с высоким retention (например, театральные постановки) — перспективные «точки входа». Для жанров с низким retention стоит разработать дополнительные стимулы: скидки, рекомендации похожих мероприятий.

3. **Региональная стратегия.** В регионах с высокой активностью, но сниженным retention стоит усилить маркетинговое присутствие и расширить ассортимент мероприятий.

4. **Стимулирование крупных заказов.** Пользователи, покупающие несколько билетов, потенциально более вовлечены. Групповые скидки и акции для компаний могут повысить удержание.

5. **Сокращение интервала возврата.** Для пользователей с 2–4 заказами важно сокращать интервал между покупками — регулярные напоминания о предстоящих мероприятиях помогут перевести их в категорию лояльных (5+).

6. **Персонализация на основе первого заказа.** Данные о первом заказе (жанр, регион, устройство) могут использоваться для построения предиктивной модели возврата и персонализированных рекомендаций.

---
*Проект выполнен в рамках курса MLDS.*

### 6. Финализация проекта и публикация в Git

Когда вы закончите анализировать данные, оформите проект, а затем опубликуйте его.

Выполните следующие действия:

1. Создайте файл `.gitignore`. Добавьте в него все временные и чувствительные файлы, которые не должны попасть в репозиторий.
2. Сформируйте файл `requirements.txt`. Зафиксируйте все библиотеки, которые вы использовали в проекте.
3. Вынести все чувствительные данные (параметры подключения к базе) в `.env`файл.
4. Проверьте, что проект запускается и воспроизводим.
5. Загрузите проект в публичный репозиторий — например, на GitHub. Убедитесь, что все нужные файлы находятся в репозитории, исключая те, что в `.gitignore`. Ссылка на репозиторий понадобится для отправки проекта на проверку. Вставьте её в шаблон проекта в тетрадке Jupyter Notebook перед отправкой проекта на ревью.

**Вставьте ссылку на проект в этой ячейке тетрадки перед отправкой проекта на ревью.**